# E05 — Material Selection & DFM Checks
*Exam tool — simplified 3-part structure. For full analysis see `06_material_selection.ipynb`.*

---

## Part 1 — Theory Recap

### 5-Step Material Selection Method

| Step | Action |
|------|--------|
| 1 | **Establish demands** — operating temperature, loads, chemical environment, regulatory requirements |
| 2 | **Translate to thresholds** — convert demands to measurable material properties (HDT, $E$, Izod, FDA) |
| 3 | **Apply qualitative filters** — eliminate materials failing thickness-independent criteria |
| 4 | **Apply quantitative filters** — eliminate materials failing thickness-dependent criteria |
| 5 | **Final selection** — choose the most cost-effective candidate meeting all criteria |

### Boolean AND Filtering
All constraints must be satisfied **simultaneously** (AND logic, not OR).

### Key Material Properties
| Property | Column | Unit | Notes |
|----------|--------|------|-------|
| HDT | `HDT_C` | °C | Heat deflection temperature; must exceed max operating T |
| Stiffness | `tensile_modulus_MPa` | MPa | For structural loads |
| Impact | `notched_izod_kJ_m2` | kJ/m² | For drop/impact resistance |
| FDA | `FDA_approved` | bool | Mandatory for food-contact parts |
| Elongation | `elongation_yield_pct` | % | For snap-fits and living hinges |

### DFM Rules
| Check | Amorphous (PC, ABS, PMMA) | Semi-crystalline (PP, POM, PA66, PE) |
|-------|--------------------------|--------------------------------------|
| Shrinkage | 0.4–0.8% (low warpage risk) | 1.2–2.5% (high warpage risk) |
| Draft angle | 0.5–3.0° typical | 1.0–3.0° (more critical) |
| Rib thickness | ≤ 60% of wall | ≤ 60% of wall |
| Wall variation | ≤ 10% | ≤ 10% |

---

## ⚠ Common Exam Pitfalls
1. **OR instead of AND** — applying constraints with OR logic results in too many survivors.
2. **`MATERIAL_DB` uses MPa** — moduli in `MATERIAL_DB` are in MPa (not Pa); set thresholds in MPa.
3. **HDT ≠ max continuous service T** — HDT is a deflection test point, typically use a safety margin.
4. **FDA filter direction** — filter OUT materials where `FDA_approved == False` when required.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Problem Inputs  (edit values here)
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd

# ── Material database (inline — edit or extend as needed) ────────────────────
MATERIAL_DB = [
    {"name": "PP homopolymer",  "flexural_modulus_MPa": 1300, "tensile_modulus_MPa": 1300, "HDT_C":  90, "notched_izod_kJ_m2":  30, "elongation_yield_pct": 10, "FDA_approved": True},
    {"name": "PP copolymer",    "flexural_modulus_MPa":  900, "tensile_modulus_MPa":  900, "HDT_C":  80, "notched_izod_kJ_m2":  50, "elongation_yield_pct": 20, "FDA_approved": True},
    {"name": "PE-HD",           "flexural_modulus_MPa":  900, "tensile_modulus_MPa":  800, "HDT_C":  80, "notched_izod_kJ_m2":  60, "elongation_yield_pct": 20, "FDA_approved": True},
    {"name": "PE-LD",           "flexural_modulus_MPa":  200, "tensile_modulus_MPa":  200, "HDT_C":  45, "notched_izod_kJ_m2": 100, "elongation_yield_pct": 30, "FDA_approved": True},
    {"name": "POM",             "flexural_modulus_MPa": 2600, "tensile_modulus_MPa": 2800, "HDT_C": 110, "notched_izod_kJ_m2":  65, "elongation_yield_pct": 15, "FDA_approved": False},
    {"name": "PC",              "flexural_modulus_MPa": 2300, "tensile_modulus_MPa": 2300, "HDT_C": 130, "notched_izod_kJ_m2":  70, "elongation_yield_pct":  6, "FDA_approved": True},
    {"name": "ABS",             "flexural_modulus_MPa": 2200, "tensile_modulus_MPa": 2000, "HDT_C":  95, "notched_izod_kJ_m2":  20, "elongation_yield_pct":  5, "FDA_approved": False},
    {"name": "PA66 dry",        "flexural_modulus_MPa": 2800, "tensile_modulus_MPa": 3000, "HDT_C": 200, "notched_izod_kJ_m2":  50, "elongation_yield_pct":  5, "FDA_approved": True},
    {"name": "PA66-GF30",       "flexural_modulus_MPa": 7500, "tensile_modulus_MPa": 8500, "HDT_C": 240, "notched_izod_kJ_m2":  80, "elongation_yield_pct":  3, "FDA_approved": True},
    {"name": "PEEK",            "flexural_modulus_MPa": 4100, "tensile_modulus_MPa": 3700, "HDT_C": 260, "notched_izod_kJ_m2":  50, "elongation_yield_pct": 30, "FDA_approved": False},
    {"name": "PEEK-GF30",       "flexural_modulus_MPa": 9500, "tensile_modulus_MPa":10000, "HDT_C": 280, "notched_izod_kJ_m2":  40, "elongation_yield_pct":  2, "FDA_approved": False},
    {"name": "PPS",             "flexural_modulus_MPa": 3800, "tensile_modulus_MPa": 3700, "HDT_C": 260, "notched_izod_kJ_m2":  25, "elongation_yield_pct":  2, "FDA_approved": False},
    {"name": "Rigid PVC",       "flexural_modulus_MPa": 3000, "tensile_modulus_MPa": 3000, "HDT_C":  70, "notched_izod_kJ_m2":   5, "elongation_yield_pct":  3, "FDA_approved": False},
]

SHRINKAGE_RATES = {
    "PP": 0.015, "PP_copolymer": 0.018, "PE_HD": 0.020, "PE_LD": 0.025,
    "PA66": 0.012, "POM": 0.020, "PC": 0.006, "ABS": 0.006,
    "PEEK": 0.004, "PPS": 0.007, "PVC": 0.003, "PA66_GF30": 0.006,
}

# ── Step 1: Operational demands (describe your problem here) ──────────────────
DEMANDS = [
    'Food-contact approval required (FDA)',
    'Continuous operating temperature up to 85 °C',
    'Drop impact resistance (Izod > 30 kJ/m²)',
    'Structural stiffness required (E > 1500 MPa)',
]

# ── Step 2: Constraint thresholds (translate demands to numbers) ───────────────
HDT_MIN_C   = 80.0    # [°C] minimum heat deflection temperature
IZOD_MIN    = 30.0    # [kJ/m²] minimum notched Izod impact
E_MIN_MPa   = 1500.0  # [MPa] minimum tensile modulus
FDA_NEEDED  = True    # True = only FDA-approved materials survive

# ── DFM parameters ────────────────────────────────────────────────────────────
WALL_MM     = 2.5     # [mm] nominal wall thickness

print('Demands:')
for i, d in enumerate(DEMANDS, 1):
    print(f'  {i}. {d}')

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Functions
# ═══════════════════════════════════════════════════════════════════════════════

# Polymer family classification by shrinkage threshold
_SEMICRYSTALLINE_PREFIXES = ('pp', 'pe', 'pom', 'pa', 'peek', 'pps', 'ptfe')


def classify_crystallinity(name):
    """Returns 'semi-crystalline' or 'amorphous' based on material name prefix."""
    n = name.lower()
    for prefix in _SEMICRYSTALLINE_PREFIXES:
        if n.startswith(prefix):
            return 'semi-crystalline'
    return 'amorphous'


def shrinkage_lookup(name, shrinkage_dict):
    """Best-effort lookup in SHRINKAGE_RATES by case-insensitive prefix matching."""
    n = name.lower()
    for key, val in shrinkage_dict.items():
        if n.startswith(key.lower()) or key.lower() in n:
            return val, key
    return None, None


def dfm_advice(name, shrinkage_dict, wall_mm):
    """Returns DFM guidance dict for a given material."""
    cryst = classify_crystallinity(name)
    shr_val, shr_key = shrinkage_lookup(name, shrinkage_dict)
    shr_pct = shr_val * 100.0 if shr_val is not None else None

    if cryst == 'semi-crystalline':
        draft   = '1.0–3.0°'
        warpage = 'HIGH' if (shr_pct or 0) >= 1.5 else 'MODERATE'
    else:
        draft   = '0.5–3.0°'
        warpage = 'LOW' if (shr_pct or 0) <= 0.8 else 'MODERATE'

    return {
        'crystallinity': cryst,
        'shrinkage_pct': shr_pct,
        'draft_angle':   draft,
        'warpage_risk':  warpage,
        'rib_max_mm':    round(wall_mm * 0.60, 2),
    }


def filter_materials(df, hdt_min, izod_min, e_min, fda_required):
    """Apply constraints progressively, print elimination counts, return survivors."""
    n_total = len(df)
    print(f'  Total materials in database : {n_total}')

    if fda_required:
        df = df[df['FDA_approved'] == True]
        print(f'  After FDA filter            : {len(df)} remain  ({n_total - len(df)} eliminated)')

    n_before = len(df)
    df = df[df['HDT_C'] >= hdt_min]
    print(f'  After HDT >= {hdt_min:.0f}°C          : {len(df)} remain  ({n_before - len(df)} eliminated)')

    n_before = len(df)
    df = df[df['notched_izod_kJ_m2'] >= izod_min]
    print(f'  After Izod >= {izod_min:.0f} kJ/m²     : {len(df)} remain  ({n_before - len(df)} eliminated)')

    n_before = len(df)
    df = df[df['tensile_modulus_MPa'] >= e_min]
    print(f'  After E >= {e_min:.0f} MPa         : {len(df)} remain  ({n_before - len(df)} eliminated)')

    return df.reset_index(drop=True)


print('Functions defined.')


Functions defined.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Execution
# ═══════════════════════════════════════════════════════════════════════════════

df_all = pd.DataFrame(MATERIAL_DB)

# --- Step 3: Progressive filtering -------------------------------------------
print('--- Step 3: Progressive Filtering ---')
df_candidates = filter_materials(df_all, HDT_MIN_C, IZOD_MIN, E_MIN_MPa, FDA_NEEDED)

# If no candidates, relax constraints and try again
if len(df_candidates) == 0:
    print()
    print('  INFO: No materials pass all constraints.')
    print(f'  Relaxing: HDT >= {HDT_MIN_C - 10:.0f}°C, E >= {E_MIN_MPa - 200:.0f} MPa')
    df_candidates = filter_materials(df_all,
                                     HDT_MIN_C - 10, IZOD_MIN,
                                     E_MIN_MPa - 200, FDA_NEEDED)

# --- Step 4: Candidates table ------------------------------------------------
print()
print('--- Step 4: Candidate Materials ---')
if len(df_candidates) == 0:
    print('  No candidates — consider further relaxing constraints.')
else:
    cols = ['name', 'tensile_modulus_MPa', 'HDT_C', 'notched_izod_kJ_m2',
            'elongation_yield_pct', 'FDA_approved']
    print(df_candidates[cols].to_string(index=False))

# --- Step 5: DFM advice per candidate ----------------------------------------
print()
print('--- Step 5: DFM Advice per Candidate ---')
print(f"  {'Material':<20} {'Type':<18} {'Shrinkage':<12} {'Draft':<12} {'Warpage':<10} {'Rib max [mm]'}")
print(f"  {'-'*20} {'-'*18} {'-'*12} {'-'*12} {'-'*10} {'-'*12}")
for _, row in df_candidates.iterrows():
    info = dfm_advice(row['name'], SHRINKAGE_RATES, WALL_MM)
    shr  = f"{info['shrinkage_pct']:.1f}%" if info['shrinkage_pct'] is not None else 'n/a'
    print(f"  {row['name']:<20} {info['crystallinity']:<18} {shr:<12}"
          f" {info['draft_angle']:<12} {info['warpage_risk']:<10} {info['rib_max_mm']}")


--- Step 3: Progressive Filtering ---
  Total materials in database : 12
  After FDA filter            : 7 remain  (5 eliminated)
  After HDT >= 80°C          : 6 remain  (1 eliminated)
  After Izod >= 30 kJ/m²     : 6 remain  (0 eliminated)
  After E >= 1500 MPa         : 3 remain  (3 eliminated)

--- Step 4: Candidate Materials ---
     name  tensile_modulus_MPa  HDT_C  notched_izod_kJ_m2  elongation_yield_pct  FDA_approved
       PC                 2300    130                  70                     6          True
 PA66 dry                 3000    200                  50                     5          True
PA66-GF30                 8500    240                  80                     3          True

--- Step 5: DFM Advice per Candidate ---
  Material             Type               Shrinkage    Draft        Warpage    Rib max [mm]
  -------------------- ------------------ ------------ ------------ ---------- ------------
  PC                   amorphous          0.6%         0.5–3.0

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Validation
# ═══════════════════════════════════════════════════════════════════════════════

n_candidates = len(df_candidates)
pass_selection = n_candidates >= 1

print('--- VALIDATION ---')
print(f"  Candidates found : {n_candidates}  (need >= 1)  |  {'PASS' if pass_selection else 'FAIL'}")

if pass_selection:
    # Simple recommendation: highest Izod (most robust for general use)
    best = df_candidates.loc[df_candidates['notched_izod_kJ_m2'].idxmax(), 'name']
    best_info = dfm_advice(best, SHRINKAGE_RATES, WALL_MM)
    print(f"  Recommendation   : {best}")
    print(f"    Crystallinity  : {best_info['crystallinity']}")
    shr_str = f"{best_info['shrinkage_pct']:.1f}%" if best_info['shrinkage_pct'] is not None else 'n/a'
    print(f"    Shrinkage      : {shr_str}")
    print(f"    Draft angle    : {best_info['draft_angle']}")
    print(f"    Warpage risk   : {best_info['warpage_risk']}")
    print(f"    Rib max        : {best_info['rib_max_mm']} mm  (60% of {WALL_MM} mm wall)")

print(f"  OVERALL          : {'PASS' if pass_selection else 'FAIL'}")


--- VALIDATION ---
  Candidates found : 3  (need >= 1)  |  PASS
  Recommendation   : PA66-GF30
    Crystallinity  : semi-crystalline
    Shrinkage      : 1.2%
    Draft angle    : 1.0–3.0°
    Warpage risk   : MODERATE
    Rib max        : 1.5 mm  (60% of 2.5 mm wall)
  OVERALL          : PASS
